In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import logging

from nichejepa.datasets import make_cell_neighborhood_dataset, prepare_dataset
from nichejepa.masks.multigene import MaskCollator
from nichejepa.utils.config import create_params_from_YAML_wandb_config

In [ ]:
world_size = 1
rank = 0

In [ ]:
logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)
params = create_params_from_YAML_wandb_config('../../../nichejepa/configs/test.yaml',
                                              logger)
params["data"]["batch_size"] = 2

In [ ]:
params

In [ ]:
train_dataset, _ = prepare_dataset(params)
train_dataset

In [ ]:
mask_collator = MaskCollator(
    n_targets=params["mask"]["n_targets"],
    n_contexts=params["mask"]["n_contexts"],
    target_mask_size=params["mask"]["target_mask_size"],
    context_mask_size=params["mask"]["context_mask_size"],
    seq_len_cell=params["data"]["seq_len_cell"],
    seq_len_neighborhood=params["data"]["seq_len_neighborhood"],
    has_cls=params["data"]["has_cls"])

In [ ]:
_, train_loader, train__sampler = make_cell_neighborhood_dataset(
            batch_size=params["data"]["batch_size"],
            data=train_dataset,
            vocab_size=params["data"]["vocab_size"],
            collator=mask_collator,
            pin_mem=params["data"]["pin_mem"],
            num_workers=params["data"]["num_workers"],
            world_size=world_size,
            rank=rank,
            drop_last=False,
            seq_len_cell=params["data"]["seq_len_cell"],
            seq_len_neighborhood=params["data"]["seq_len_neighborhood"],
            has_cls=params["data"]["has_cls"])

In [ ]:
for itr, (udata, masks_enc, masks_pred, masks_attention) in enumerate(train_loader):
    print(udata)
    print(masks_enc)
    print(masks_pred)
    print(masks_attention)
    break